In [ ]:
import re, os
from pathlib import Path              # ✅ Correct import for filesystem Path
import pandas as pd
import matplotlib.pyplot as plt
import georinex as gr
import warnings

DATA_DIR = Path(r"P:\2026\BHP Solitude Spillway\260619\01_CAPTURE\01_SURVEYING\04_BENCHMARK\RINEX")

# Suppress xarray FutureWarnings about join/compat changes to avoid clutter
warnings.filterwarnings("ignore", category=FutureWarning, message=".*the default value for join will change.*")

# Find RINEX observation files (RINEX 2 or 3, possibly compressed)
obs_files = []
for f in DATA_DIR.glob("*"):
    if f.is_file():
        fname = f.name
        # Remove compression extension (so detection works on .YYo, .obs, etc.)
        if fname.lower().endswith((".gz", ".z", ".zip", ".bz2")):  # include .bz2 as well
            fname = os.path.splitext(fname)[0]
        # Check typical RINEX obs file extensions: .YYo/.YYd (RINEX2) or .obs/.rnx/.crx (RINEX3)
        if re.match(r".*\.[0-9]{2}[oOdD]$", fname) or fname.lower().endswith((".obs", ".rnx", ".crx")):
            obs_files.append(f)
obs_files.sort()
if not obs_files:
    raise FileNotFoundError(f"No RINEX observation files found in {DATA_DIR}")

obs_path = obs_files[0]  # Use first RINEX file (adjust logic if multiple files should be processed)

obs_data = gr.load(obs_path)  # Load observation data as xarray Dataset
times = pd.to_datetime(obs_data.time.values)  # observation times as pandas datetimes

# Identify unique GNSS constellations from satellite IDs
sat_ids = [sv.decode('utf-8') if isinstance(sv, bytes) else str(sv) for sv in obs_data.sv.values]
constellations = sorted({sid[0] for sid in sat_ids})

constellation_names = {
    'G': 'GPS', 'R': 'GLONASS', 'E': 'Galileo', 'C': 'BeiDou',
    'J': 'QZSS', 'S': 'SBAS', 'I': 'IRNSS'
}
color_map = {
    'G': 'blue', 'R': 'red', 'E': 'green', 'C': 'orange',
    'J': 'purple', 'S': 'magenta', 'I': 'brown'
}

# Count satellites per epoch for each constellation
constellation_counts = {c: [] for c in constellations}
for t in obs_data.time:
    # Drop satellites with no data at this epoch (dropna with how='all' drops an sv if *all* its obs types are NaN)
    epoch_data = obs_data.sel(time=t).dropna(dim='sv', how='all')
    active_sats = [sv.decode('utf-8') if isinstance(sv, bytes) else str(sv) for sv in epoch_data.sv.values]
    for c in constellations:
        count = sum(1 for sv in active_sats if sv.startswith(c))
        constellation_counts[c].append(count)

# Alternatively, a vectorized approach (faster for large datasets):
# obs_array = obs_data.to_array()  # combine obs types into one DataArray
# sat_present = obs_array.notnull().any(dim='variable')  # True for each (time, sv) where any obs is present
# total_counts = sat_present.sum(dim='sv').values  # total satellites count at each epoch
# For per-constellation counts, you could use similar logic by filtering sat_present by sv index groups.

# Compute flight duration in minutes for x-axis
minutes = (times - times[0]).total_seconds() / 60.0

# Determine y-axis max (max satellites observed at any epoch, plus one)
max_count = max(max(cnt_list) for cnt_list in constellation_counts.values())
y_max = max_count + 1

# Plot satellite availability by constellation over time
plt.figure(figsize=(10, 6))
for c in constellations:
    # Only plot constellations that have at least one satellite observed
    if any(constellation_counts[c]):
        plt.plot(minutes, constellation_counts[c],
                 label=constellation_names.get(c, c),
                 color=color_map.get(c, None))
plt.xlabel("Flight Duration (minutes)")
plt.ylabel("Number of Satellites")
plt.ylim(0, y_max)                    # Start y-axis at 0 and cap at max+1
plt.title(f"Satellite Availability by Constellation\n"
          f"{times[0].strftime('%Y-%m-%d')} | {times[0].strftime('%H:%M:%S')} to {times[-1].strftime('%H:%M:%S')} UTC")
plt.legend()
plt.grid(True)
plt.show()